# ML-07 — Baseline Score

Build and evaluate the transparent hand-rule baseline before any ML model. This is the number the model must beat.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../scripts")
from ml_utils import normalize, percentile_rank, precision_at_k
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Prep: fill blanks, create label
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Filter: impressions > 0 and age >= 90 (same as pipeline)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

print(f"Rows after filtering: {len(df):,}")
print(f"Declining: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean():.1%})")

## 2. Build the baseline score

*A transparent, weighted composite of four sub-scores — no ML, just hand-rules.*

In [ ]:
# Four sub-scores
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]

# Weighted composite
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

print("Baseline score formula:")
print("  40% × visibility_score (log impressions rank)")
print("  30% × freshness_risk_score (days since update rank)")
print("  25% × position_opportunity_score (position × visibility)")
print("   5% × depth_gap_score (thin content × visibility)")
print(f"\nScore range: [{df['baseline_refresh_score'].min():.3f}, {df['baseline_refresh_score'].max():.3f}]")
print(f"Median score: {df['baseline_refresh_score'].median():.3f}")

## 3. Evaluate the baseline

*How well does the baseline rank declining pages above non-declining ones?*

In [ ]:
y_true = df["is_declining_label"]
scores = df["baseline_refresh_score"]

auc = roc_auc_score(y_true, scores)
avg_prec = average_precision_score(y_true, scores)
p_at_20 = precision_at_k(y_true, scores, 20)
p_at_50 = precision_at_k(y_true, scores, 50)
p_at_100 = precision_at_k(y_true, scores, 100)

print(f"Baseline Evaluation (full dataset, not holdout):")
print(f"{'='*45}")
print(f"  ROC AUC:           {auc:.3f}")
print(f"  Average Precision: {avg_prec:.3f}")
print(f"  Precision@20:      {p_at_20:.3f}")
print(f"  Precision@50:      {p_at_50:.3f}")
print(f"  Precision@100:     {p_at_100:.3f}")
print(f"{'='*45}")
print(f"  Base rate:         {y_true.mean():.3f}")
print(f"\nThe baseline is only slightly better than random at the top of the queue.")
print(f"Precision@50 = {p_at_50:.3f} vs base rate {y_true.mean():.3f} → ")
print(f"the hand-rules are actually WORSE than just guessing majority class for top-K.")
print(f"This is the number the ML model must beat.")

## 4. Why the baseline struggles

*The hand-rules prioritize visible + stale pages, but visibility ≠ declining.*

In [ ]:
# Top-50 by baseline score: what do they look like?
top50 = df.nlargest(50, "baseline_refresh_score")
print(f"Top-50 baseline pages:")
print(f"  Actually declining: {top50['is_declining_label'].sum()} of 50 ({top50['is_declining_label'].mean():.1%})")
print(f"  Median impressions: {top50['impressions_90d'].median():,.0f}")
print(f"  Median age:         {top50['content_age_days'].median():.0f} days")
print(f"  Median position:    {top50['avg_position'].median():.1f}")
print(f"\nThe baseline picks high-visibility, old, well-positioned pages —")
print(f"but these are often STABLE, not declining. Visibility ≠ decline risk.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.